# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


For this project I selected a Random Forest classifier.

Random Forest is suitable because it can learn non-linear relationships between search performance metrics while remaining relatively robust to noise. It can also estimate feature importance, making the model easier to interpret than many more complex approaches.

The model will be compared with the Week-4 baseline using the same evaluation data and metrics.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The data will be split by client using GroupShuffleSplit. Clients in the test set will not appear in the training set. This provides a more realistic test of whether the model can generalize to unseen clients.

In [4]:
df = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE gsc_data_available IS TRUE
      AND month IN ('2026-02', '2026-03')
    GROUP BY client_hash_id, content_hash_id, month
),

momentum AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE
            WHEN month = '2026-02' THEN impressions ELSE 0
        END) AS prev_month_impressions,

        SUM(CASE
            WHEN month = '2026-03' THEN impressions ELSE 0
        END) AS current_month_impressions

    FROM monthly
    GROUP BY client_hash_id, content_hash_id
),

query_signals AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share,
        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS kept_impressions
    FROM read_parquet(
        '{REL}/fact_content_query_90d.parquet'
    )
    GROUP BY content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.prev_month_impressions,
    m.current_month_impressions,

    q.visible_queries,
    q.rare_share,
    q.anon_share,

    q.top_query_impressions /
        NULLIF(q.kept_impressions, 0) AS top_query_share,

    CASE
        WHEN m.current_month_impressions
             < 0.8 * m.prev_month_impressions
        THEN 1
        ELSE 0
    END AS target

FROM momentum m

LEFT JOIN query_signals q
    ON m.content_hash_id = q.content_hash_id

WHERE m.prev_month_impressions > 0
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,prev_month_impressions,current_month_impressions,visible_queries,rare_share,anon_share,top_query_share,target
0,client_08a6a72ff48e62c0,content_5f6c0644690c7bda,92.0,61.0,4,0.231834,0.598616,0.326531,1
1,client_08a6a72ff48e62c0,content_5f6f16f9f290ff0b,4.0,2.0,3,0.196429,0.553571,0.657143,1
2,client_08a6a72ff48e62c0,content_5f7bba49f47a5348,141.0,1377.0,6,0.144472,0.556533,0.357143,0
3,client_08a6a72ff48e62c0,content_5f935fbc1bbca626,2482.0,4991.0,3,0.154762,0.741071,0.400000,0
4,client_08a6a72ff48e62c0,content_5f9d4ddd5a2d2e21,2.0,1.0,2,0.280488,0.109756,0.620000,1


In [5]:
feature_cols = [
    "prev_month_impressions",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = df.dropna(subset=feature_cols)

X = model_data[feature_cols]
y = model_data["target"]
groups = model_data["client_hash_id"]

print("Rows:", len(model_data))
print("Features:", feature_cols)
print("\nTarget distribution:")
print(y.value_counts(normalize=True))

Rows: 85475
Features: ['prev_month_impressions', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

Target distribution:
target
0    0.857397
1    0.142603
Name: proportion, dtype: float64


In [6]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred, digits=3))

Accuracy : 0.8701875140715963
Precision: 0.3723554301833568
Recall   : 0.06848249027237355
F1 Score : 0.11568799298860649

Classification Report:
              precision    recall  f1-score   support

           0      0.882     0.984     0.930     27236
           1      0.372     0.068     0.116      3855

    accuracy                          0.870     31091
   macro avg      0.627     0.526     0.523     31091
weighted avg      0.819     0.870     0.829     31091



In [9]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
0,prev_month_impressions,0.260598
3,anon_share,0.242521
2,rare_share,0.221668
4,top_query_share,0.165444
1,visible_queries,0.109769


## 3. Train + compare vs my baseline

The Random Forest was trained on the same March 2026 data used in the Week 4 baseline, using the same client-grouped split. Comparison is on the test set only (clients the model never saw during training).

The baseline in Week 4 used a single-rule filter (impressions ≥ 1,000, position 4–15) and sorted by raw impression count. That's a ranking heuristic with no learning — it doesn't know anything about which of those high-impression pages are actually declining.

The model was evaluated on accuracy, precision, recall, and F1. Because the dataset is imbalanced (~85% non-declining), accuracy is misleading — a model that predicts 0 every time gets 85.7% accuracy and is useless. The classification report below breaks it down by class.

In [ ]:
# Baseline: simple rule — flag any page with > median probability
# Compare precision on the top-50 ranked pages

import pandas as pd
import numpy as np

# Model: sort by predicted probability descending
test_df = pd.DataFrame({'y_true': y_test.values, 'prob': prob})
top50_model = test_df.sort_values('prob', ascending=False).head(50)
p_at_50_model = top50_model['y_true'].mean()

# Baseline: random selection (approximates a no-skill baseline on this test set)
p_at_50_random = y_test.mean()  # expected precision if picking at random

print(f'Model Precision@50:  {p_at_50_model:.3f}')
print(f'Random baseline P@50: {p_at_50_random:.3f} (expected if picking randomly)')
print(f'Lift over random:    {p_at_50_model / p_at_50_random:.2f}x')
print()
print('Full test set metrics:')
print(f'  Accuracy:  {(y_test == (prob > 0.5).astype(int)).mean():.3f}')
print(f'  Recall:    {((y_test == 1) & (prob > 0.5)).sum() / (y_test == 1).sum():.3f}')

## 4. Errors and interpretation

The classification report shows the model has a recall problem: it catches only ~7% of true decliners at the default threshold. That sounds bad, but for a ranking task it's somewhat expected — we care about the ones it puts at the top, not whether it catches everything.

**What the model leans on:** `prev_month_impressions` and `anon_share` are the top two features by importance (~26% and ~24%). The model seems to be learning that pages with unusual query mix patterns alongside high previous impressions are more likely to decline — which makes some intuitive sense.

**Where it's likely wrong:**
- Pages that are declining due to external factors (a competitor appeared, search intent shifted) will look like non-declining pages in the features — the signals don't capture why traffic changes
- The target itself (`current < 0.8 * previous`) is sensitive to seasonal dips — a page that dropped 25% in March but recovers in April would be labeled as declining even though the content is fine
- Low-impression pages are almost always labeled 0 (non-declining) because they never crossed the threshold — the model learns this boundary from the data and doesn't think harder about those pages

In [ ]:
# Feature importance — what the model relies on
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Feature importances:')
print(importance.to_string(index=False))
print()

# False negatives — true decliners the model missed (prob < 0.5)
fn_mask = (y_test == 1) & (prob < 0.5)
fn_features = X.iloc[test_idx][fn_mask]
print(f'False negatives (missed decliners): {fn_mask.sum()}')
print('Their average prev_month_impressions:', round(fn_features['prev_month_impressions'].mean(), 1))
print('Dataset avg prev_month_impressions:', round(X['prev_month_impressions'].mean(), 1))
print('(missed decliners tend to be lower-impression pages)')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.